# XGBoost 기반 인터넷 서비스 수요 예측 모델

## 1. 프로젝트 개요

### 배경
- Wealth Effect (Modigliani, 1966): 집값 상승 → 자산가치 증가 → 소비심리 확대
- 한국 가구 자산의 70~80%가 부동산 → 부동산 심리가 소비에 직접 영향
- 부동산 공포/탐욕 지수(FGI)를 직접 설계하여 인터넷 렌탈 서비스 수요와의 관계를 분석

### Target
- CONTRACT_COUNT: 인터넷 서비스 계약 건수

### 데이터셋에 존재하는 컬럼 (학습에 직접 사용하지 않음)
| Column | Description | 비고 |
|--------|-------------|------|
| YEAR_MONTH | 기준 연월 | |
| STATE / CITY | 시도 / 시군구 | 인코딩 처리 (전처리용) |
| MOVEMENT_COUNT / RATE | 순이동 수 / 률 | FGI 구성요소 |
| MEAN_MEME_PRICE / MEAN_JEONSE_PRICE | 평균 매매가 / 전세가 | |
| CHANGE_MEME_PRICE_RATE | 매매가 전월 변동률 | FGI 구성요소 |
| REVERSE_JEONSE_PER_MEME | 매매가 대비 전세가 역비율 | FGI 구성요소 |
| RATE_GAP / PRICE_MOMENTUM_3M / PRICE_VOLATILITY_6M | 괴리율 / 모멘텀 / 변동성 | FGI 구성요소 |
| FEAR_GREED_INDEX | 공포탐욕지수 | |

### 학습 Feature (계산 생성, 14개)
| Feature | 설명 |
|---------|------|
| FGI_LAG1 / LAG2 / LAG3 / LAG6 | 1/2/3/6개월 전 공포탐욕지수 |
| CONTRACT_COUNT_LAG1 / LAG2 / LAG3 / LAG6 | 1/2/3/6개월 전 계약 건수 |
| CHANGE_MEME_PRICE_RATE_LAG1 / LAG2 / LAG3 / LAG6 | 1/2/3/6개월 전 매매가 변동률 |
| MONTH_SIN / MONTH_COS | 월 주기성 sin/cos 변환 |

### NULL 처리 3가지 데이터셋
| Dataset | 방법 | Rows |
|---------|------|------|
| REMOVE | NULL 행 삭제 | 4,842 |
| MEAN | 평균값 대체 | 7,704 |
| ZERO | 0으로 대체 | 7,704 |

## 2. Data Load & Feature Engineering

In [ ]:
%%sql -r df_all
SELECT 'REMOVE' AS DATASET, * FROM HAKERTON_KAKAO4.PUBLIC.CALC_FGI_INTERNET_NN_REMOVE
UNION ALL
SELECT 'MEAN' AS DATASET, * FROM HAKERTON_KAKAO4.PUBLIC.CALC_FGI_INTERNET_NN_MEAN
UNION ALL
SELECT 'ZERO' AS DATASET, * FROM HAKERTON_KAKAO4.PUBLIC.CALC_FGI_INTERNET_NN
ORDER BY DATASET, STATE, CITY, YEAR_MONTH

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from itertools import product as iter_product
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

def build_features(df_in):
    df = df_in.copy()
    df['YEAR_MONTH'] = pd.to_datetime(df['YEAR_MONTH'])
    df = df.sort_values(['STATE', 'CITY', 'YEAR_MONTH'])
    le_s, le_c = LabelEncoder(), LabelEncoder()
    df['STATE'] = le_s.fit_transform(df['STATE'])
    df['CITY'] = le_c.fit_transform(df['CITY'])
    m = df['YEAR_MONTH'].dt.month
    df['MONTH_SIN'] = np.sin(2 * np.pi * m / 12)
    df['MONTH_COS'] = np.cos(2 * np.pi * m / 12)
    for lag in [1, 2, 3, 6]:
        df[f'FGI_LAG{lag}'] = df.groupby(['STATE', 'CITY'])['FEAR_GREED_INDEX'].shift(lag)
        df[f'CONTRACT_COUNT_LAG{lag}'] = df.groupby(['STATE', 'CITY'])['CONTRACT_COUNT'].shift(lag)
        df[f'CHANGE_MEME_PRICE_RATE_LAG{lag}'] = df.groupby(['STATE', 'CITY'])['CHANGE_MEME_PRICE_RATE'].shift(lag)
    return df

FEAT_WITH_LAG = [
    'FGI_LAG1', 'FGI_LAG2', 'FGI_LAG3', 'FGI_LAG6',
    'CONTRACT_COUNT_LAG1', 'CONTRACT_COUNT_LAG2', 'CONTRACT_COUNT_LAG3', 'CONTRACT_COUNT_LAG6',
    'CHANGE_MEME_PRICE_RATE_LAG1', 'CHANGE_MEME_PRICE_RATE_LAG2',
    'CHANGE_MEME_PRICE_RATE_LAG3', 'CHANGE_MEME_PRICE_RATE_LAG6',
    'MONTH_SIN', 'MONTH_COS'
]
FEAT_NO_LAG = [f for f in FEAT_WITH_LAG if 'CONTRACT_COUNT_LAG' not in f]
TARGET = 'CONTRACT_COUNT'

datasets = {}
for name in ['REMOVE', 'MEAN', 'ZERO']:
    raw = df_all[df_all['DATASET'] == name].drop(columns=['DATASET'])
    datasets[name] = build_features(raw)

print(f"Features with CONTRACT_COUNT_LAG: {len(FEAT_WITH_LAG)}")
print(f"Features without CONTRACT_COUNT_LAG: {len(FEAT_NO_LAG)}")
for name, df in datasets.items():
    n = df.dropna(subset=FEAT_WITH_LAG + [TARGET]).shape[0]
    print(f"{name}: {n} rows (after dropna)")

## 3. XGBoost 모델 학습: 3개 데이터셋 비교 (CONTRACT_COUNT_LAG 포함)

14개 Feature를 사용하여 3개 NULL 처리 데이터셋별 XGBoost 모델을 학습하고,하이퍼파라미터 튜닝을 통해 과적합을 줄인 후 성능을 비교한다.

In [ ]:
def train_and_tune(df, features, target, name):
    df_m = df.dropna(subset=features + [target]).sort_values('YEAR_MONTH')
    sp = int(len(df_m) * 0.8)
    tr, te = df_m.iloc[:sp], df_m.iloc[sp:]
    Xtr, ytr = tr[features], tr[target]
    Xte, yte = te[features], te[target]
    
    mdl_before = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42)
    mdl_before.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
    yp_b = mdl_before.predict(Xte)
    before = {
        'Train_R2': r2_score(ytr, mdl_before.predict(Xtr)),
        'Test_R2': r2_score(yte, yp_b),
        'RMSE': np.sqrt(mean_squared_error(yte, yp_b)),
        'MAE': mean_absolute_error(yte, yp_b)
    }
    before['Gap'] = before['Train_R2'] - before['Test_R2']
    
    param_grid = {
        'max_depth': [3, 4, 5], 'n_estimators': [200, 300],
        'learning_rate': [0.03, 0.05], 'min_child_weight': [10, 20, 30],
        'subsample': [0.6, 0.7], 'reg_alpha': [1.0, 5.0], 'reg_lambda': [5.0, 10.0]
    }
    best_gap, best_params = 999, None
    for combo in iter_product(*param_grid.values()):
        p = dict(zip(param_grid.keys(), combo))
        m = XGBRegressor(colsample_bytree=0.6, random_state=42, **p)
        m.fit(Xtr, ytr, verbose=False)
        tr2 = r2_score(ytr, m.predict(Xtr))
        te2 = r2_score(yte, m.predict(Xte))
        g = tr2 - te2
        if te2 > 0.9 and g < best_gap:
            best_gap, best_params = g, p
            best_te, best_tr = te2, tr2
    
    mdl_tuned = XGBRegressor(colsample_bytree=0.6, random_state=42, **best_params)
    mdl_tuned.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
    yp_t = mdl_tuned.predict(Xte)
    after = {
        'Train_R2': r2_score(ytr, mdl_tuned.predict(Xtr)),
        'Test_R2': r2_score(yte, yp_t),
        'RMSE': np.sqrt(mean_squared_error(yte, yp_t)),
        'MAE': mean_absolute_error(yte, yp_t)
    }
    after['Gap'] = after['Train_R2'] - after['Test_R2']
    
    return {'before': before, 'after': after, 'model': mdl_tuned,
            'Xte': Xte, 'yte': yte, 'yp': yp_t, 'params': best_params, 'N': len(df_m)}

results_with = {}
for name in ['REMOVE', 'MEAN', 'ZERO']:
    print(f"Training {name}...")
    results_with[name] = train_and_tune(datasets[name], FEAT_WITH_LAG, TARGET, name)

print("\n=== 3 Dataset Comparison (CONTRACT_COUNT_LAG included) ===")
print(f"{'Dataset':>10s} {'N':>6s} | {'Before R²':>10s} {'Before Gap':>10s} | {'Tuned R²':>10s} {'Tuned Gap':>10s} {'RMSE':>8s} {'MAE':>8s}")
print('-' * 85)
for name in ['REMOVE', 'MEAN', 'ZERO']:
    r = results_with[name]
    best = ' ←' if r['after']['Test_R2'] == max(v['after']['Test_R2'] for v in results_with.values()) else ''
    print(f"{name:>10s} {r['N']:>6d} | {r['before']['Test_R2']:>10.4f} {r['before']['Gap']:>10.4f} | {r['after']['Test_R2']:>10.4f} {r['after']['Gap']:>10.4f} {r['after']['RMSE']:>8.2f} {r['after']['MAE']:>8.2f}{best}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, name in enumerate(['REMOVE', 'MEAN', 'ZERO']):
    r = results_with[name]
    
    axes[0][i].scatter(r['yte'], r['yp'], alpha=0.3, s=10, color=['#4a90d9', '#7fb069', '#e8873d'][i])
    mn, mx = min(r['yte'].min(), r['yp'].min()), max(r['yte'].max(), r['yp'].max())
    axes[0][i].plot([mn, mx], [mn, mx], 'r--', lw=1)
    axes[0][i].set_xlabel('Actual')
    axes[0][i].set_ylabel('Predicted')
    axes[0][i].set_title(f"{name} (Tuned R²={r['after']['Test_R2']:.4f})")
    
    imps = pd.Series(r['model'].feature_importances_, index=FEAT_WITH_LAG).sort_values(ascending=True)
    colors = ['red' if 'CONTRACT_COUNT_LAG' in f else 'coral' if 'FGI' in f else 'gold' if 'CHANGE_MEME' in f else 'steelblue' for f in imps.index]
    imps.plot(kind='barh', ax=axes[1][i], color=colors)
    axes[1][i].set_title(f"{name}: Feature Importance")

plt.suptitle('3 Dataset Comparison (with CONTRACT_COUNT_LAG)\nRed=ContractLAG, Orange=FGI, Yellow=PriceRate, Blue=Time', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n=== Feature Category Summary ===")
print(f"{'Dataset':>10s} {'Contract%':>12s} {'FGI%':>8s} {'Price%':>8s} {'Time%':>8s}")
for name in ['REMOVE', 'MEAN', 'ZERO']:
    imps = pd.Series(results_with[name]['model'].feature_importances_, index=FEAT_WITH_LAG)
    c = sum(imps[f] for f in imps.index if 'CONTRACT_COUNT_LAG' in f)
    g = sum(imps[f] for f in imps.index if 'FGI' in f)
    p = sum(imps[f] for f in imps.index if 'CHANGE_MEME' in f)
    t = sum(imps[f] for f in imps.index if 'MONTH' in f)
    print(f"{name:>10s} {c:>12.1%} {g:>8.1%} {p:>8.1%} {t:>8.1%}")

## 4. CONTRACT_COUNT_LAG 제거 모델

위 모델에서 CONTRACT_COUNT_LAG가 95~97%를 지배하고 있다.
아정당 서비스가 매년 급성장(2023: 80건 → 2024: 388건 → 2025: 719건) 중이기 때문에,
과거 계약수가 미래를 거의 그대로 예측하는 것은 당연한 결과이다.

"부동산 지표(FGI)가 서비스 수요에 영향을 미치는가"를 검증하려면
CONTRACT_COUNT_LAG를 제거하고 순수하게 FGI 관련 Feature만으로 학습해야 한다.

In [ ]:
def train_and_tune_nolag(df, features, target, name):
    df_m = df.dropna(subset=features + [target]).sort_values('YEAR_MONTH')
    sp = int(len(df_m) * 0.8)
    tr, te = df_m.iloc[:sp], df_m.iloc[sp:]
    Xtr, ytr = tr[features], tr[target]
    Xte, yte = te[features], te[target]
    
    mdl_b = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42)
    mdl_b.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
    yp_b = mdl_b.predict(Xte)
    before = {'Train_R2': r2_score(ytr, mdl_b.predict(Xtr)), 'Test_R2': r2_score(yte, yp_b),
              'RMSE': np.sqrt(mean_squared_error(yte, yp_b)), 'MAE': mean_absolute_error(yte, yp_b)}
    before['Gap'] = before['Train_R2'] - before['Test_R2']
    
    param_grid = {
        'max_depth': [2, 3, 4], 'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.03, 0.05], 'min_child_weight': [20, 30, 50],
        'subsample': [0.5, 0.6], 'reg_alpha': [5.0, 10.0], 'reg_lambda': [10.0, 20.0]
    }
    best_gap, best_params, best_te = 999, None, -999
    for combo in iter_product(*param_grid.values()):
        p = dict(zip(param_grid.keys(), combo))
        m = XGBRegressor(colsample_bytree=0.5, random_state=42, **p)
        m.fit(Xtr, ytr, verbose=False)
        tr2 = r2_score(ytr, m.predict(Xtr))
        te2 = r2_score(yte, m.predict(Xte))
        g = tr2 - te2
        if te2 > best_te and g < 0.5:
            best_gap, best_te, best_params = g, te2, p
    
    mdl_t = XGBRegressor(colsample_bytree=0.5, random_state=42, **best_params)
    mdl_t.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
    yp_t = mdl_t.predict(Xte)
    after = {'Train_R2': r2_score(ytr, mdl_t.predict(Xtr)), 'Test_R2': r2_score(yte, yp_t),
             'RMSE': np.sqrt(mean_squared_error(yte, yp_t)), 'MAE': mean_absolute_error(yte, yp_t)}
    after['Gap'] = after['Train_R2'] - after['Test_R2']
    return {'before': before, 'after': after, 'model': mdl_t, 'Xte': Xte, 'yte': yte, 'yp': yp_t, 'N': len(df_m)}

results_no = {}
for name in ['REMOVE', 'MEAN', 'ZERO']:
    print(f"Training {name} (no CONTRACT_COUNT_LAG)...")
    results_no[name] = train_and_tune_nolag(datasets[name], FEAT_NO_LAG, TARGET, name)

print("\n=== 3 Dataset Comparison (CONTRACT_COUNT_LAG removed, 10 features) ===")
print(f"{'Dataset':>10s} {'N':>6s} | {'Before R²':>10s} {'Before Gap':>10s} | {'Tuned R²':>10s} {'Tuned Gap':>10s}")
print('-' * 72)
for name in ['REMOVE', 'MEAN', 'ZERO']:
    r = results_no[name]
    print(f"{name:>10s} {r['N']:>6d} | {r['before']['Test_R2']:>10.4f} {r['before']['Gap']:>10.4f} | {r['after']['Test_R2']:>10.4f} {r['after']['Gap']:>10.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(['REMOVE', 'MEAN', 'ZERO']):
    r = results_no[name]
    imps = pd.Series(r['model'].feature_importances_, index=FEAT_NO_LAG).sort_values(ascending=True)
    colors = ['coral' if 'FGI' in f else 'gold' if 'CHANGE_MEME' in f else 'steelblue' for f in imps.index]
    imps.plot(kind='barh', ax=axes[i], color=colors)
    fgi_s = sum(imps[f] for f in imps.index if 'FGI' in f)
    axes[i].set_title(f"{name}: FGI={fgi_s:.1%} (R²={r['after']['Test_R2']:.4f})")
plt.suptitle('Feature Importance without CONTRACT_COUNT_LAG\nOrange=FGI, Yellow=PriceRate, Blue=Time', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n=== Feature Category (No LAG, Tuned) ===")
print(f"{'Dataset':>10s} {'FGI%':>8s} {'Price%':>8s} {'Time%':>8s}")
for name in ['REMOVE', 'MEAN', 'ZERO']:
    imps = pd.Series(results_no[name]['model'].feature_importances_, index=FEAT_NO_LAG)
    g = sum(imps[f] for f in imps.index if 'FGI' in f)
    p = sum(imps[f] for f in imps.index if 'CHANGE_MEME' in f)
    t = sum(imps[f] for f in imps.index if 'MONTH' in f)
    print(f"{name:>10s} {g:>8.1%} {p:>8.1%} {t:>8.1%}")

## 5. 지역 그룹별 모델

지역 규모 차이를 통제하기 위해 **수도권(서울/경기/인천)**, **광역시(부산/대구/대전/광주/울산/세종)**, **지방**으로 분리하여 모델링한다.
REMOVE 데이터셋 기준, CONTRACT_COUNT_LAG 포함 14개 Feature로 학습.

In [ ]:
df_region_raw = df_all[df_all['DATASET'] == 'REMOVE'].drop(columns=['DATASET']).copy()
df_region_raw['YEAR_MONTH'] = pd.to_datetime(df_region_raw['YEAR_MONTH'])
metro_list = ['서울', '경기', '인천']
city_list = ['부산', '대구', '대전', '광주', '울산', '세종']
def get_region(s):
    if s in metro_list: return 'Metro'
    elif s in city_list: return 'City'
    else: return 'Rural'
df_region_raw['REGION'] = df_region_raw['STATE'].apply(get_region)

reg_results = []
for region in ['Metro', 'City', 'Rural']:
    df_r = df_region_raw[df_region_raw['REGION'] == region].copy().drop(columns=['REGION'])
    df_r = build_features(df_r)
    df_r_m = df_r.dropna(subset=FEAT_WITH_LAG + [TARGET]).sort_values('YEAR_MONTH')
    sp = int(len(df_r_m) * 0.8)
    tr_r, te_r = df_r_m.iloc[:sp], df_r_m.iloc[sp:]
    Xtr_r, ytr_r = tr_r[FEAT_WITH_LAG], tr_r[TARGET]
    Xte_r, yte_r = te_r[FEAT_WITH_LAG], te_r[TARGET]
    
    m_r = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.7, min_child_weight=5, reg_alpha=0.1, reg_lambda=1.0, random_state=42)
    m_r.fit(Xtr_r, ytr_r, eval_set=[(Xte_r, yte_r)], verbose=False)
    yp_r = m_r.predict(Xte_r)
    tr2_r = r2_score(ytr_r, m_r.predict(Xtr_r))
    te2_r = r2_score(yte_r, yp_r)
    
    imps_r = pd.Series(m_r.feature_importances_, index=FEAT_WITH_LAG)
    c_imp = sum(imps_r[f] for f in imps_r.index if 'CONTRACT_COUNT_LAG' in f)
    g_imp = sum(imps_r[f] for f in imps_r.index if 'FGI' in f)
    p_imp = sum(imps_r[f] for f in imps_r.index if 'CHANGE_MEME' in f)
    t_imp = sum(imps_r[f] for f in imps_r.index if 'MONTH' in f)
    
    reg_results.append({'Region': region, 'N': len(df_r_m), 'Test_R2': te2_r, 'Gap': tr2_r-te2_r,
        'Contract': c_imp, 'FGI': g_imp, 'Price': p_imp, 'Time': t_imp,
        'model': m_r, 'Xte': Xte_r, 'yte': yte_r, 'yp': yp_r})

print("=== Regional Model Results ===")
print(f"{'Region':>8s} {'N':>6s} {'Test R²':>10s} {'Gap':>8s} | {'Contract':>10s} {'FGI':>8s} {'Price':>8s} {'Time':>8s}")
print('-' * 75)
for r in reg_results:
    print(f"{r['Region']:>8s} {r['N']:>6d} {r['Test_R2']:>10.4f} {r['Gap']:>8.4f} | {r['Contract']:>10.1%} {r['FGI']:>8.1%} {r['Price']:>8.1%} {r['Time']:>8.1%}")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, r in enumerate(reg_results):
    axes[0][i].scatter(r['yte'], r['yp'], alpha=0.3, s=10)
    mn, mx = min(r['yte'].min(), r['yp'].min()), max(r['yte'].max(), r['yp'].max())
    axes[0][i].plot([mn, mx], [mn, mx], 'r--', lw=1)
    axes[0][i].set_title(f"{r['Region']} (R²={r['Test_R2']:.4f})")
    axes[0][i].set_xlabel('Actual'); axes[0][i].set_ylabel('Predicted')
    
    imps = pd.Series(r['model'].feature_importances_, index=FEAT_WITH_LAG).sort_values(ascending=True)
    colors = ['red' if 'CONTRACT_COUNT_LAG' in f else 'coral' if 'FGI' in f else 'gold' if 'CHANGE_MEME' in f else 'steelblue' for f in imps.index]
    imps.plot(kind='barh', ax=axes[1][i], color=colors)
    axes[1][i].set_title(f"{r['Region']}: Feature Importance")
plt.suptitle('Regional Models (Red=ContractLAG, Orange=FGI, Yellow=PriceRate, Blue=Time)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. 전체 모델 비교 종합

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

names = ['REMOVE', 'MEAN', 'ZERO']
with_r2 = [results_with[n]['after']['Test_R2'] for n in names]
no_r2 = [results_no[n]['after']['Test_R2'] for n in names]

x = np.arange(len(names))
w = 0.35
axes[0][0].bar(x - w/2, with_r2, w, label='With LAG (14 feat)', color='steelblue')
axes[0][0].bar(x + w/2, no_r2, w, label='Without LAG (10 feat)', color='coral')
axes[0][0].set_xticks(x)
axes[0][0].set_xticklabels(names)
axes[0][0].set_ylabel('Test R²')
axes[0][0].set_title('Test R² by Dataset & Model')
axes[0][0].legend()
axes[0][0].axhline(y=0, color='black', lw=0.5)
for i, (v1, v2) in enumerate(zip(with_r2, no_r2)):
    axes[0][0].text(i - w/2, v1 + 0.01, f'{v1:.3f}', ha='center', fontsize=9)
    axes[0][0].text(i + w/2, max(v2, 0) + 0.01, f'{v2:.3f}', ha='center', fontsize=9)

with_gap = [results_with[n]['after']['Gap'] for n in names]
no_gap = [results_no[n]['after']['Gap'] for n in names]
axes[0][1].bar(x - w/2, with_gap, w, label='With LAG', color='steelblue')
axes[0][1].bar(x + w/2, no_gap, w, label='Without LAG', color='coral')
axes[0][1].set_xticks(x)
axes[0][1].set_xticklabels(names)
axes[0][1].set_ylabel('Gap (Train R² - Test R²)')
axes[0][1].set_title('Overfitting Gap by Dataset & Model')
axes[0][1].legend()
axes[0][1].axhline(y=0.1, color='red', lw=0.5, linestyle='--', label='Gap=0.1')

reg_names = [r['Region'] for r in reg_results]
reg_r2 = [r['Test_R2'] for r in reg_results]
reg_fgi = [r['FGI'] * 100 for r in reg_results]
axes[1][0].bar(reg_names, reg_r2, color=['#4a90d9', '#7fb069', '#e8873d'])
axes[1][0].set_ylabel('Test R²')
axes[1][0].set_title('Regional Model R²')
for i, v in enumerate(reg_r2):
    axes[1][0].text(i, v + 0.01, f'{v:.4f}', ha='center')

categories = ['Contract\nLAG', 'FGI\nLAG', 'Price Rate\nLAG', 'Time']
for i, r in enumerate(reg_results):
    vals = [r['Contract'] * 100, r['FGI'] * 100, r['Price'] * 100, r['Time'] * 100]
    offset = (i - 1) * 0.25
    axes[1][1].bar([j + offset for j in range(len(categories))], vals, 0.25,
                   label=r['Region'], color=['#4a90d9', '#7fb069', '#e8873d'][i])
axes[1][1].set_xticks(range(len(categories)))
axes[1][1].set_xticklabels(categories)
axes[1][1].set_ylabel('Importance (%)')
axes[1][1].set_title('Feature Category by Region')
axes[1][1].legend()

plt.suptitle('Model Comparison Summary', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"\n{'Model':>35s} {'Test R²':>10s} {'Gap':>8s} {'RMSE':>10s} {'MAE':>10s}")
print('-' * 75)
for name in names:
    r = results_with[name]
    print(f"{'With LAG - ' + name:>35s} {r['after']['Test_R2']:>10.4f} {r['after']['Gap']:>8.4f} {r['after']['RMSE']:>10.2f} {r['after']['MAE']:>10.2f}")
print('-' * 75)
for name in names:
    r = results_no[name]
    print(f"{'No LAG - ' + name:>35s} {r['after']['Test_R2']:>10.4f} {r['after']['Gap']:>8.4f} {r['after']['RMSE']:>10.2f} {r['after']['MAE']:>10.2f}")
print('-' * 75)
for r in reg_results:
    print(f"{'Regional - ' + r['Region']:>35s} {r['Test_R2']:>10.4f} {r['Gap']:>8.4f}")

## 7. 결론

### 모델 성능 요약

CONTRACT_COUNT_LAG 포함 시:
- 3개 데이터셋 모두 R²=0.97~0.98로 높은 예측 성능
- 하이퍼파라미터 튜닝으로 과적합(Gap)을 0.002~0.01 수준까지 감소
- 그러나 CONTRACT_COUNT_LAG가 Feature Importance의 92~97%를 지배
- 이는 아정당 서비스의 사업 성장 트렌드를 반영한 것이지, FGI의 영향이 아님

CONTRACT_COUNT_LAG 제거 시:
- R²가 음수로 하락 → FGI와 매매가 변동률만으로는 계약 건수 예측 불가
- 그러나 Feature Importance에서 FGI가 32~52%로 가장 중요한 변수로 부상
- FGI는 "예측"은 못 하지만, 남은 변수 중에서는 가장 높은 관련성을 가짐

지역 그룹별 분석:
- 수도권(R²=0.96), 지방(R²=0.94), 광역시(R²=0.87) 순으로 예측 성능
- 광역시에서 FGI+매매변동률의 영향이 상대적으로 가장 큼(~5.4%)

### 핵심 인사이트

> 인터넷 서비스 수요를 예측하려면 과거 계약 데이터(CONTRACT_COUNT_LAG)가 필수적이며,
> FGI 단독으로는 예측이 불가능하다. 그러나 FGI는 보조 지표로서 약 2~52%의 설명력을 가지며,
> 특히 광역시 지역에서 부동산 심리와 서비스 수요의 관련성이 상대적으로 높게 나타났다.

---

## 8. 한계점 및 향후 과제

### 한계점

1. 데이터 기간 부족
- 아정당 서비스 데이터가 2023년부터 존재하여 분석 기간이 약 2.5년으로 제한적
- 부동산 사이클(5~10년)을 온전히 관찰하기에 부족하며, FGI 변동폭이 좁아(878~897) 공포/탐욕 구간을 명확히 구분하기 어려움
- 2022년 금리 급등기 같은 하락장(공포 구간) 데이터가 포함되지 않아 실증 분석이 불완전

2. 사업 성장 트렌드와 FGI 효과 분리 어려움
- 아정당 계약수가 매년 급성장(2023: 80건 → 2024: 388건 → 2025: 719건)하여 CONTRACT_COUNT_LAG가 모델의 92~97%를 지배
- CONTRACT_COUNT_LAG를 제거하면 R²가 음수 → FGI 단독으로는 예측 불가
- 계약 증가가 FGI 때문인지, 사업 자체의 성장 때문인지 구분이 난해

3. 인과관계 미증명
- FGI가 높은 지역에서 계약이 많다는 상관은 확인했으나, "FGI 때문에 계약이 늘었다"는 인과는 미증명
- 대도시라는 공통 원인이 FGI와 계약 모두를 높였을 가능성 존재

4. 단일 서비스 카테고리 분석
- 인터넷 서비스만 분석하여, "공포일 때 필수재 수요 증가 / 탐욕일 때 사치재 수요 증가"라는 가설의 비교 검증이 불완전

### 향후 과제

1. 데이터 축적: 아정당 데이터 3~5년 축적 시 부동산 사이클 전체를 분석할 수 있으며, FGI의 예측력도 강화될 것으로 기대
2. 외부 변수 추가: 기준금리, 대출 규제, 신규 입주 물량 등 거시경제 변수 추가
3. 서비스 카테고리 간 비교: 인터넷(필수재) vs 정수기/안마의자(사치재)의 FGI 반응 차이 비교
4. FGI 가중치 최적화: 현재 임의 설정된 가중치를 데이터 기반 회귀 분석으로 재산출
5. 지역 세분화 모델 고도화: 수도권/광역시/지방별 개별 모델을 튜닝하여 지역 특성에 맞는 수요 예측 체계 구축